# 3 — Best Model Selection & Refinement
## FireSpreadNet · Next Day Wildfire Spread

**Workflow:**
1. Load all training results from previous notebooks
2. **Auto-select the best model** by test Dice (fallback: val Dice)
3. **Fine-tune** the selected model with a small LR
4. **Final evaluation** with optimal threshold + AUC-PR
5. Save `results/best_model_selection.json` → used by Notebook 05 (XAI)

This notebook is hardware-agnostic: it adapts batch size to available VRAM.


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Setup
# ══════════════════════════════════════════════════════════════
import gc, json, os, sys, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from sklearn.metrics import precision_recall_curve, auc as sklearn_auc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)
mpl.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 150})
%matplotlib inline

sys.path.insert(0, os.path.abspath('..'))

from config import MODEL_CONFIG, TRAIN_CONFIG, SEED, MODELS_DIR, RESULTS_DIR, FIGURES_DIR
from src.data.dataset import get_dataloaders
from src.models.cellular_automata import CellularAutomataModel
from src.models.convlstm import ConvLSTMModel
from src.models.unet import UNetFire
from src.models.pi_cca import PIConvCellularAutomaton
from src.training.trainer import CombinedLoss, compute_metrics

_cfg_path = Path().resolve() / 'setup_config.json'
if not _cfg_path.exists():
    raise FileNotFoundError('setup_config.json not found — run 00_Setup.ipynb first')
cfg = json.load(open(_cfg_path))

PROCESSED_DIR    = Path(cfg['PROCESSED_DIR'])
FEATURE_CHANNELS = cfg['FEATURE_CHANNELS']
N_INPUT_CHANNELS = cfg['N_INPUT_CHANNELS']
CH               = cfg['CH']
norm_stats       = cfg['norm_stats']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)')
else:
    vram_gb = 0
    print('CPU mode')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

MODEL_CLASSES = {
    'ca':       CellularAutomataModel,
    'convlstm': ConvLSTMModel,
    'unet':     UNetFire,
    'pi_cca':   PIConvCellularAutomaton,
}

print(f'Device: {device}')


## 3.1 Data Loading

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Data Loading
# ══════════════════════════════════════════════════════════════
BS = 8 if vram_gb < 8 else 16

loaders = get_dataloaders(
    PROCESSED_DIR,
    batch_size=BS,
    num_workers=0,       # num_workers=0 required on Windows
    augment_train=True,
    seed=SEED,
    stats=norm_stats,
)

for split, loader in loaders.items():
    print(f'{split}: {len(loader.dataset):,} samples  ({len(loader)} batches, bs={BS})')

x_batch, y_batch = next(iter(loaders['train']))
print(f'\nBatch: X={x_batch.shape}  Y={y_batch.shape}')
print(f'Fire pixel ratio (train batch): {y_batch.mean():.4f}')


## 3.2 Auto-Select Best Model

Scans all result files produced by training notebooks and picks the model
with the best **test Dice** (fallback: val Dice if test results are unavailable).

Priority order for result files:
1. `saved_models/{model}/eval_results_camber.json` (Camber run)
2. `saved_models/{model}/eval_results.json` (local run)
3. `results/all_results_camber.json` / `results/all_results_local.json`
4. Training history val Dice as last resort

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — Auto-select best model from all available results
# ══════════════════════════════════════════════════════════════
def _scan_model_results(model_name):
    """Return (test_dice, val_dice, source_file) for model_name."""
    per_model_files = [
        MODELS_DIR / model_name / 'eval_results_camber.json',
        MODELS_DIR / model_name / 'eval_results.json',
    ]
    global_files = [
        (RESULTS_DIR / 'all_results_camber.json', model_name),
        (RESULTS_DIR / 'all_results_local.json',  model_name),
    ]

    best_test, best_val, source = None, None, 'none'

    # 1. Per-model eval files
    for p in per_model_files:
        if not p.exists():
            continue
        with open(p) as f:
            r = json.load(f)
        t_dice = r.get('test', r.get('val', {})).get('dice')
        v_dice = r.get('val', {}).get('dice')
        if t_dice is not None and (best_test is None or t_dice > best_test):
            best_test, best_val, source = t_dice, v_dice, p.name

    # 2. Global results files
    for gpath, key in global_files:
        if not gpath.exists():
            continue
        with open(gpath) as f:
            all_r = json.load(f)
        if key not in all_r:
            continue
        r = all_r[key]
        t_dice = r.get('test', r.get('val', {})).get('dice')
        v_dice = r.get('val', {}).get('dice')
        if t_dice is not None and (best_test is None or t_dice > best_test):
            best_test, best_val, source = t_dice, v_dice, gpath.name

    # 3. Fallback: val dice from training history
    if best_test is None:
        for hname in ['training_history_camber.json', 'training_history.json']:
            hp = MODELS_DIR / model_name / hname
            if not hp.exists():
                continue
            with open(hp) as f:
                h = json.load(f)
            if 'val_metrics' in h and h['val_metrics']:
                best_val = max(m.get('dice', 0) for m in h['val_metrics'])
                best_test = best_val          # use val as proxy
                source = hname + ' (val only)'
                break

    return best_test, best_val, source


rows = []
for name in ['ca', 'convlstm', 'unet', 'pi_cca']:
    n_params = sum(p.numel() for p in MODEL_CLASSES[name](config=MODEL_CONFIG[name]).parameters()
                   if p.requires_grad)
    ckpt     = MODELS_DIR / name / 'best_model.pt'
    td, vd, src = _scan_model_results(name)
    rows.append({
        'Model':      MODEL_CONFIG[name]['name'],
        'key':        name,
        'Params':     f'{n_params:,}',
        'Checkpoint': '✓' if ckpt.exists() else '✗',
        'Test Dice':  f'{td:.4f}' if td else 'N/A',
        'Val Dice':   f'{vd:.4f}' if vd else 'N/A',
        'Source':     src,
        '_score':     td if td else (vd or 0.0),
    })

df_all = pd.DataFrame(rows).drop(columns=['key', '_score'])
print('=' * 70)
print('  ALL TRAINED MODELS')
print('=' * 70)
display(df_all)

# ── Select best ──
valid = [r for r in rows
         if r['_score'] > 0 and (MODELS_DIR / r['key'] / 'best_model.pt').exists()]

if not valid:
    raise RuntimeError(
        'No trained model checkpoint found.\n'
        'Run 03_Training_Local_GTX1050.ipynb or 03_Training_Camber.ipynb first.'
    )

best_entry     = max(valid, key=lambda r: r['_score'])
BEST_MODEL_KEY = best_entry['key']

print(f'\n★ Auto-selected: {MODEL_CONFIG[BEST_MODEL_KEY]["name"]}  '
      f'(score={best_entry["_score"]:.4f}, source={best_entry["Source"]})')


## 3.3 Fine-tune Best Model

Load the best checkpoint and continue training for a short period with a
**reduced learning rate** (cosine from 5e-5 → 1e-7) and early stopping.

- **Hardware-adaptive**: uses AMP on GPU, CPU fallback
- **Memory-safe**: running confusion-matrix metrics (no pred accumulation)
- **Saves** the improved checkpoint to `saved_models/{model}/best_model.pt`

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — Refinement trainer (memory-safe, AMP, running metrics)
# ══════════════════════════════════════════════════════════════
class RefinementTrainer:
    """
    Short fine-tuning loop:
    - Cosine LR from fine_lr down to 1e-7
    - Linear warmup (3 epochs)
    - Running confusion-matrix metrics → no GPU pred accumulation
    - AMP on CUDA
    """

    def __init__(self, model, model_name, device, fine_lr=5e-5,
                 weight_decay=1e-4, focal_alpha=0.75, focal_gamma=2.0,
                 use_amp=True, accum_steps=1):
        self.model       = model.to(device)
        self.model_name  = model_name
        self.device      = device
        self.use_amp     = use_amp and torch.cuda.is_available()
        self.accum_steps = accum_steps
        self.criterion   = CombinedLoss(
            focal_w=0.5, dice_w=0.5, alpha=focal_alpha, gamma=focal_gamma)
        params = [p for p in model.parameters() if p.requires_grad]
        self.has_params = bool(params)
        self.optimizer  = (torch.optim.AdamW(params, lr=fine_lr,
                                             weight_decay=weight_decay)
                           if params else None)
        self.scaler  = GradScaler() if self.use_amp else None
        self.history = {'train_loss': [], 'val_loss': [],
                        'train_dice': [], 'val_dice': [], 'lr': []}

    @staticmethod
    def _dice_from_cm(tp, fp, fn):
        return 2 * tp / (2 * tp + fp + fn + 1e-8)

    def _cosine_lr(self, epoch, total, warmup, base_lr):
        if epoch < warmup:
            return base_lr * (epoch + 1) / warmup
        prog = (epoch - warmup) / max(1, total - warmup)
        return 1e-7 + 0.5 * (base_lr - 1e-7) * (1 + np.cos(np.pi * prog))

    def train(self, train_loader, val_loader, epochs=20, patience=7,
              warmup_epochs=3, grad_clip=1.0):
        base_lr = (self.optimizer.param_groups[0]['lr']
                   if self.optimizer else 0.0)
        best_dice, no_improve = 0.0, 0
        save_dir = MODELS_DIR / self.model_name
        save_dir.mkdir(parents=True, exist_ok=True)
        print(f'\n{"="*60}')
        print(f'Fine-tuning: {MODEL_CONFIG[self.model_name]["name"]}')
        print(f'LR={base_lr:.1e} | AMP={self.use_amp} | Epochs={epochs} | Patience={patience}')
        print(f'{"="*60}')
        for epoch in range(epochs):
            t0  = time.time()
            lr  = self._cosine_lr(epoch, epochs, warmup_epochs, base_lr)
            if self.optimizer:
                for pg in self.optimizer.param_groups:
                    pg['lr'] = lr
            if self.has_params:
                tl, td = self._train_epoch(train_loader, grad_clip)
            else:
                tl, td = self._eval_epoch(train_loader)
            vl, vd = self._eval_epoch(val_loader)
            dt = time.time() - t0
            for k, v in [('train_loss', tl), ('val_loss', vl),
                         ('train_dice', td), ('val_dice', vd), ('lr', lr)]:
                self.history[k].append(float(v))
            print(f'Epoch {epoch+1:2d}/{epochs} | '
                  f'Loss {tl:.4f}/{vl:.4f} | '
                  f'Dice {td:.4f}/{vd:.4f} | '
                  f'LR={lr:.2e} | {dt:.1f}s')
            if vd > best_dice:
                best_dice, no_improve = vd, 0
                torch.save(self.model.state_dict(), save_dir / 'best_model.pt')
            else:
                no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}  (best Dice={best_dice:.4f})')
                break
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
        print(f'\nBest val Dice after refinement: {best_dice:.4f}')
        return self.history

    def _train_epoch(self, loader, grad_clip):
        self.model.train()
        total_loss = 0; tp = fp = fn = 0
        self.optimizer.zero_grad()
        for step, (x, y) in enumerate(tqdm(loader, desc='Refine', leave=False)):
            x = x.to(self.device, non_blocking=True)
            y = y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                    loss = self.criterion(pred.float(), y.float()) / self.accum_steps
                self.scaler.scale(loss).backward()
                if (step + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y) / self.accum_steps
                loss.backward()
                if (step + 1) % self.accum_steps == 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
            total_loss += loss.item() * self.accum_steps * x.size(0)
            with torch.no_grad():
                pb = (pred.detach() > 0.5).float(); tb = y.float()
                tp += (pb * tb).sum().item()
                fp += (pb * (1 - tb)).sum().item()
                fn += ((1 - pb) * tb).sum().item()
        return total_loss / len(loader.dataset), self._dice_from_cm(tp, fp, fn)

    @torch.no_grad()
    def _eval_epoch(self, loader):
        self.model.eval()
        total_loss = 0; tp = fp = fn = 0
        for x, y in tqdm(loader, desc='Eval', leave=False):
            x = x.to(self.device, non_blocking=True)
            y = y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                    loss = self.criterion(pred.float(), y.float())
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            pb = (pred > 0.5).float(); tb = y.float()
            tp += (pb * tb).sum().item()
            fp += (pb * (1 - tb)).sum().item()
            fn += ((1 - pb) * tb).sum().item()
        return total_loss / len(loader.dataset), self._dice_from_cm(tp, fp, fn)


# ── Load best checkpoint & launch fine-tuning ──
accum = 2 if BEST_MODEL_KEY == 'pi_cca' and vram_gb < 8 else 1
fine_lr = 5e-5                     # 6× lower than typical training LR

model = MODEL_CLASSES[BEST_MODEL_KEY](config=MODEL_CONFIG[BEST_MODEL_KEY]).to(device)

ckpt_path = MODELS_DIR / BEST_MODEL_KEY / 'best_model.pt'
if ckpt_path.exists():
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    print(f'Loaded pre-trained weights from {ckpt_path}')
else:
    print(f'No checkpoint found at {ckpt_path} — training from scratch')

trainer = RefinementTrainer(
    model, BEST_MODEL_KEY, device,
    fine_lr=fine_lr, weight_decay=1e-4,
    focal_alpha=0.75, focal_gamma=2.0,
    use_amp=True, accum_steps=accum,
)
refinement_history = trainer.train(
    loaders['train'], loaders['val'],
    epochs=20, patience=7, warmup_epochs=3, grad_clip=1.0,
)

with open(MODELS_DIR / BEST_MODEL_KEY / 'refinement_history.json', 'w') as f:
    json.dump(refinement_history, f, indent=2, default=str)
print('\nRefinement history saved.')


## 3.4 Final Evaluation (Threshold Opt + AUC-PR)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 — Full evaluation of refined model
# ══════════════════════════════════════════════════════════════
@torch.no_grad()
def full_evaluate(model, val_loader, test_loader, device, use_amp=True,
                  thresholds=None):
    """Threshold sweep on val + AUC-PR + final test metrics."""
    use_amp_eff = use_amp and torch.cuda.is_available()
    model.eval()

    # ── Collect val predictions (on CPU to avoid OOM) ──
    val_preds, val_targets = [], []
    for x, y in tqdm(val_loader, desc='Val eval', leave=False):
        x = x.to(device, non_blocking=True)
        if use_amp_eff:
            with autocast('cuda'):
                pred = model(x)
        else:
            pred = model(x)
        val_preds.append(pred.float().cpu())
        val_targets.append(y.float())
    val_preds   = torch.cat(val_preds)
    val_targets = torch.cat(val_targets)

    # ── Optimal threshold sweep ──
    if thresholds is None:
        thresholds = np.arange(0.05, 0.90, 0.025)
    best_dice, best_t = 0.0, 0.5
    for t in thresholds:
        pb = (val_preds > t).float(); tb = val_targets.float()
        tp = (pb * tb).sum().item(); fp = (pb*(1-tb)).sum().item()
        fn = ((1-pb)*tb).sum().item()
        dice = 2*tp/(2*tp+fp+fn+1e-8)
        if dice > best_dice:
            best_dice, best_t = dice, float(t)

    # ── Metrics at optimal threshold (val) ──
    def _metrics(preds, targets, t):
        pb = (preds > t).float(); tb = targets.float()
        tp = (pb*tb).sum().item(); fp = (pb*(1-tb)).sum().item()
        fn = ((1-pb)*tb).sum().item(); tn = ((1-pb)*(1-tb)).sum().item()
        eps = 1e-8
        prec = tp/(tp+fp+eps); rec = tp/(tp+fn+eps)
        dice = 2*tp/(2*tp+fp+fn+eps); iou = tp/(tp+fp+fn+eps)
        f1   = 2*prec*rec/(prec+rec+eps)
        spec = tn/(tn+fp+eps)
        # AUC-PR
        p_np = preds.numpy().flatten(); t_np = targets.numpy().flatten()
        prec_c, rec_c, _ = precision_recall_curve(t_np, p_np)
        auc_pr = sklearn_auc(rec_c, prec_c)
        return {'dice': dice, 'iou': iou, 'f1': f1,
                'precision': prec, 'recall': rec,
                'specificity': spec, 'auc_pr': auc_pr,
                'threshold': t}

    val_metrics = _metrics(val_preds, val_targets, best_t)
    print(f'Optimal threshold: {best_t:.3f}  (val Dice={best_dice:.4f})')

    # ── Test predictions ──
    test_preds, test_targets = [], []
    for x, y in tqdm(test_loader, desc='Test eval', leave=False):
        x = x.to(device, non_blocking=True)
        if use_amp_eff:
            with autocast('cuda'):
                pred = model(x)
        else:
            pred = model(x)
        test_preds.append(pred.float().cpu())
        test_targets.append(y.float())
    test_preds   = torch.cat(test_preds)
    test_targets = torch.cat(test_targets)
    test_metrics = _metrics(test_preds, test_targets, best_t)

    return {
        'val':               val_metrics,
        'test':              test_metrics,
        'optimal_threshold': best_t,
        'val_preds':         val_preds,
        'val_targets':       val_targets,
        'test_preds':        test_preds,
        'test_targets':      test_targets,
    }


# Load refined checkpoint
state = torch.load(MODELS_DIR / BEST_MODEL_KEY / 'best_model.pt', map_location=device)
model.load_state_dict(state)
model.eval()

eval_results = full_evaluate(
    model, loaders['val'], loaders['test'], device, use_amp=True)

# ── Save eval results ──
save_eval = {k: v for k, v in eval_results.items()
             if k not in ('val_preds', 'val_targets', 'test_preds', 'test_targets')}
with open(MODELS_DIR / BEST_MODEL_KEY / 'eval_results_refined.json', 'w') as f:
    json.dump(save_eval, f, indent=2, default=str)

# ── Print final table ──
print('\n' + '='*60)
print(f'  FINAL RESULTS — {MODEL_CONFIG[BEST_MODEL_KEY]["name"]}')
print('='*60)
for split, m in [('Val', eval_results['val']), ('Test', eval_results['test'])]:
    print(f'\n  {split}:  Dice={m["dice"]:.4f}  IoU={m["iou"]:.4f}  '
          f'F1={m["f1"]:.4f}  AUC-PR={m["auc_pr"]:.4f}')
    print(f'       Precision={m["precision"]:.4f}  '
          f'Recall={m["recall"]:.4f}  Spec={m["specificity"]:.4f}')


## 3.5 Save Selection & Visualizations

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — Save best_model_selection.json (read by XAI notebook)
# ══════════════════════════════════════════════════════════════
selection = {
    'best_model':         BEST_MODEL_KEY,
    'best_model_name':    MODEL_CONFIG[BEST_MODEL_KEY]['name'],
    'optimal_threshold':  eval_results['optimal_threshold'],
    'val_metrics':        eval_results['val'],
    'test_metrics':       eval_results['test'],
}
with open(RESULTS_DIR / 'best_model_selection.json', 'w') as f:
    json.dump(selection, f, indent=2, default=str)
print(f'Saved: {RESULTS_DIR / "best_model_selection.json"}')

# ══════════════════════════════════════════════════════════════
# Cell 6b — Refinement training curves
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ep = range(1, len(refinement_history['train_loss']) + 1)
c_train, c_val = '#2196F3', '#FF5722'

axes[0].plot(ep, refinement_history['train_loss'], color=c_train, label='Train')
axes[0].plot(ep, refinement_history['val_loss'],   color=c_val, linestyle='--', label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Refinement Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(ep, refinement_history['train_dice'], color=c_train, label='Train')
axes[1].plot(ep, refinement_history['val_dice'],   color=c_val, linestyle='--', label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].set_title('Refinement Dice', fontweight='bold')
axes[1].legend()

axes[2].plot(ep, refinement_history['lr'], color='#4CAF50')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Schedule (Warmup + Cosine)', fontweight='bold')
axes[2].set_yscale('log')

plt.suptitle(f'Refinement — {MODEL_CONFIG[BEST_MODEL_KEY]["name"]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'refinement_curves_{BEST_MODEL_KEY}.png',
            dpi=300, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════
# Cell 6c — Precision-Recall curve
# ══════════════════════════════════════════════════════════════
p_c, r_c, _ = precision_recall_curve(
    eval_results['test_targets'].numpy().flatten(),
    eval_results['test_preds'].numpy().flatten(),
)
auc_pr = eval_results['test']['auc_pr']

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(r_c, p_c, color='#E91E63', linewidth=2, label=f'AUC-PR = {auc_pr:.4f}')
ax.axhline(eval_results['test']['precision'], color='grey', linestyle=':',
           label=f'Precision @ τ={eval_results["optimal_threshold"]:.2f}')
ax.axvline(eval_results['test']['recall'], color='grey', linestyle='--',
           label=f'Recall @ τ={eval_results["optimal_threshold"]:.2f}')
ax.set_xlabel('Recall', fontsize=12); ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curve — {MODEL_CONFIG[BEST_MODEL_KEY]["name"]}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'pr_curve_{BEST_MODEL_KEY}.png', dpi=300, bbox_inches='tight')
plt.show()

# ══════════════════════════════════════════════════════════════
# Cell 6d — Qualitative predictions
# ══════════════════════════════════════════════════════════════
model.eval()
x_test, y_test = next(iter(loaders['test']))
n_samples = min(4, x_test.shape[0])
opt_t = eval_results['optimal_threshold']

fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

titles = ['Input Fire (t)', 'Prediction', 'Binary (τ opt)', 'Ground Truth']
for row in range(n_samples):
    fire_in = x_test[row, CH['prev_fire_mask']].numpy()
    gt      = y_test[row].squeeze().numpy()
    with torch.no_grad():
        xi = x_test[row:row+1].to(device)
        if torch.cuda.is_available():
            with autocast('cuda'):
                pred_raw = model(xi)
        else:
            pred_raw = model(xi)
    pred_np = pred_raw.float().squeeze().cpu().numpy()

    ims = [fire_in, pred_np, (pred_np > opt_t).astype(float), gt]
    cmaps = ['hot', 'RdYlBu_r', 'hot', 'hot']
    for col in range(4):
        axes[row, col].imshow(ims[col], cmap=cmaps[col], vmin=0, vmax=1)
        if row == 0:
            axes[row, col].set_title(titles[col], fontweight='bold', fontsize=11)
        axes[row, col].axis('off')

plt.suptitle(f'Test Predictions — {MODEL_CONFIG[BEST_MODEL_KEY]["name"]} (τ={opt_t:.2f})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'refined_predictions_{BEST_MODEL_KEY}.png',
            dpi=300, bbox_inches='tight')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Summary

1. **Auto-selection**: compared all available results files across training sessions; best model picked by test Dice.
2. **Fine-tuning**: 20 epochs max, cosine LR from 5e-5, early stopping (patience=7).
3. **Evaluation**: threshold sweep on val set → optimal τ; AUC-PR computed on test set.
4. **Outputs**:
   - `saved_models/{model}/best_model.pt` — refined weights
   - `saved_models/{model}/eval_results_refined.json` — metrics
   - `results/best_model_selection.json` — used by Notebook 05 (XAI)
   - `results/figures/refinement_curves_{model}.png`
   - `results/figures/pr_curve_{model}.png`
   - `results/figures/refined_predictions_{model}.png`

→ Run **Notebook 05 (XAI)** to explain the selected model.